# Llama 3.1 8B · low-noise uncertainty evaluation
**Seed 123 / training version v1 · standard, clipping only, σ = 0.1, 0.25, 0.5 · development only**

Uses the five completed adapters from Notebook 05. No retraining.

First run: select Colab L4, run Installation, restart the session, then run Sections 1–5. Send `llama31_8b_low_noise_uq_review_bundle.zip` for AI-assisted review. After restoring reviewed CSVs, run Section 1 and Section 6 only to calculate metrics.

Generates 10,000 samples (five conditions × 200 questions × ten samples), scores saved greedy answers, and computes semantic entropy using the unchanged pinned NLI model and clustering method. Sampling seed remains 123 per question as in the earlier experiment. Known clustering errors remain a limitation; this notebook does not silently change or correct the baseline method.

Training input: `checkpoints/llama31_8b_low_noise_pilot/seed123_v1`.
Outputs: `results/llama31_8b_low_noise_uq_seed123_v1`.
Previous pilot outputs remain separate. Caches reject changes in their bindings and resume by question.

The review bundle excludes condition mappings and uncertainty scores. Existing decisions can be reused only after matching question and answer identities; new labels remain explicitly AI-assisted. Human audit and clustering sensitivity analysis are still outstanding.

Section 6 reports error-detection AUROC, AP, AURC and exploratory coverage at a 10% observed error target. Thresholds are selected and evaluated on development data, not held-out data. Epsilon values come from the new adapters' accounting summaries. Compare against this run's matched controls first; historical runs had different sampled training batches.


## Installation · once, then restart session

In [ ]:
%pip install -q "transformers==4.46.3" "tokenizers==0.20.3" "peft==0.13.2" "accelerate==1.1.1" "huggingface-hub==0.26.2" "safetensors==0.4.5" "sentencepiece==0.2.0" "scikit-learn==1.6.1" "numpy<3" "pandas<3" "matplotlib<4"
# Restart session now; continue at Section 1.

## 1 · Setup and bind the low-noise adapters
Keep SEED=123 and TRAIN_VERSION="v1" for the completed low-noise report. This cell validates the condition list, model files and saved greedy answers.


In [ ]:
from google.colab import drive
from pathlib import Path
import json,os,hashlib,gc,math,random,secrets,zipfile,time
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import display

drive.mount('/content/drive')
PROJECT=Path('/content/drive/MyDrive/dp_uq_experiment')
SEED=789
TRAIN_VERSION='v1'
TRAIN=PROJECT/'checkpoints/llama31_8b_low_noise_pilot'/f'seed{SEED}_{TRAIN_VERSION}'
OUT=PROJECT/'results'/f'llama31_8b_low_noise_uq_seed{SEED}_{TRAIN_VERSION}'
OUT.mkdir(parents=True,exist_ok=True)
PRE=PROJECT/'results/llama31_8b_preflight_v1'
CONDITIONS=['standard_poisson','clipping_poisson','dp_sigma_0p1','dp_sigma_0p25','dp_sigma_0p5']
SCORES=['mean_token_entropy','mean_answer_nll','discrete_semantic_entropy']

def read(p):return json.loads(p.read_text())
def save(v,p):
    t=p.with_suffix(p.suffix+'.tmp');t.write_text(json.dumps(v,ensure_ascii=False,indent=2,allow_nan=False));os.replace(t,p)
def fp(v):return hashlib.sha256(json.dumps(v,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def fh(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def immutable(v,p):
    if p.exists():assert read(p)==v, f'Identity changed: {p.name}. Restore original settings; do not reuse mismatched caches.'
    else:save(v,p)
def records_path(c):return OUT/(c+'_records.json')
def semantic_path(c):return OUT/(c+'_semantic.json')
config=read(TRAIN/'suite_config.json');common=config['common']
assert common['initialization_seed']==SEED
assert common['run_mode']=='llama_low_noise_development_pilot'
assert {r['condition']:r['noise_multiplier'] for r in config['conditions']}==dict(zip(CONDITIONS,[0.0,0.0,0.1,0.25,0.5]))
summaries={r['condition']:r for r in read(TRAIN/'suite_summary.json')}
assert set(summaries)==set(CONDITIONS)
assert all(r['optimizer_steps']==63 and r['adapter_reload_verified'] and r['all_finite_checks_passed'] for r in summaries.values())
assert len({r['initial_adapter_hash'] for r in summaries.values()})==1
MODEL_ID=common['model_id'];REVISION=common['model_revision']
assert MODEL_ID=='meta-llama/Llama-3.1-8B'
shots=common['public_demonstrations'];assert len(shots)==5
splits=read(PROJECT/'data/qa_eval_splits_v1.json')
assert fp(splits)==common['evaluation_split_hash']
dev=splits['dev']; assert len(dev)==200 and len({r['question_id'] for r in dev})==200
adapters={c:TRAIN/c/'adapter' for c in CONDITIONS}
adapter_hashes={c:fh(adapters[c]/'adapter_model.safetensors') for c in CONDITIONS}
adapter_configs={c:fh(adapters[c]/'adapter_config.json') for c in CONDITIONS}
greedy={};greedy_hashes={}
for c in CONDITIONS:
    p=TRAIN/'development'/(c+'_generations.json');cache=read(p)
    assert cache['binding']['adapter_file_hash']==adapter_hashes[c]
    assert cache['binding']['adapter_config_hash']==adapter_configs[c]
    assert cache['binding']['dev_hash']==fp(dev)
    assert cache['binding']['suite_config_hash']==fp(config)
    assert cache['binding']['model_revision']==REVISION
    greedy[c]={r['question_id']:r for r in cache['records']}
    assert len(cache['records'])==len(greedy[c])==200
    for item in dev:assert greedy[c][item['question_id']]['source_hash']==fp(item)
    greedy_hashes[c]=fh(p)
versions={p:metadata.version(p) for p in ['torch','transformers','peft','accelerate','tokenizers','sentencepiece','scikit-learn','numpy','pandas']}
NLI_ID='MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
previous_pin=PROJECT/'results/matched_base_lora_dev200_v1/nli_revision.json'
if (OUT/'nli_revision.json').exists():nli_pin=read(OUT/'nli_revision.json')
else:
    assert previous_pin.exists(), 'Restore the pinned NLI revision from your Qwen Notebook 03.'
    nli_pin=read(previous_pin);save(nli_pin,OUT/'nli_revision.json')
assert nli_pin['model_id']==NLI_ID and nli_pin['revision']
NLI_REV=nli_pin['revision']
identity={'model_id':MODEL_ID,'revision':REVISION,'suite_config_hash':fp(config),
    'adapter_hashes':adapter_hashes,'adapter_config_hashes':adapter_configs,'greedy_file_hashes':greedy_hashes,
    'dev_hash':fp(dev),'demonstrations':shots,'samples':10,'temperature':.7,'top_p':.9,'top_k':0,
    'max_new_tokens':32,'sample_seed':'sha256(123:question_id) first8 hex; reset per question',
    'scoring':'saved greedy token prefix; exclude EOS and trailing whitespace tokens; retain mixed newline token',
    'nli':nli_pin,'clustering':'all_members_bidirectional_entailment_v1','versions':versions}
immutable(identity,OUT/'evaluation_identity.json');BINDING=fp(identity)
print(f'Using low-noise seed{SEED}_{TRAIN_VERSION}','| Questions:',len(dev),'| Output:',OUT)


## 2 · Export blinded review bundle
Identical predictions for the same question/reference share one review row across all five conditions. No uncertainty, condition or initial EM label is included. Private mapping stays outside the reviewer ZIP.

Fill every question’s `decision` with `include` or `hold`, add reasons for holds and reviewer identity. Reuse documented prior validity judgments where applicable; do not infer exactly seven holds without matching IDs. For included answers fill `reviewed_correct` (0/1), `reviewer_type` (`human` or `AI-assisted`), `reviewer`, `review_note`, and `needs_adjudication` (0/1). A flagged answer requires `human_correct`, `human_reviewer`, and `human_note`. Human decisions override initial labels. All sampled answers remain diagnostic; only greedy answers are graded.

In [ ]:
salt_path=OUT/'review_salt_private.json'
if not salt_path.exists():save({'salt':secrets.token_hex(32)},salt_path)
salt=read(salt_path)['salt']
expected={};mapping=[]
for c in CONDITIONS:
    for item in dev:
        row=greedy[c][item['question_id']]
        aliases=list(dict.fromkeys([item['reference_answer'],*item['accepted_answers']]))
        key=[item['question_id'],item['question'],row['prediction'],item['reference_answer'],aliases]
        rid=fp([salt,key])
        expected[rid]={'review_id':rid,'question_id':item['question_id'],'question':item['question'],
            'prediction':row['prediction'],'reference_answer':item['reference_answer'],
            'accepted_answers':json.dumps(aliases,ensure_ascii=False)}
        mapping.append({'condition':c,'question_id':item['question_id'],'review_id':rid})
expected_review=pd.DataFrame([expected[k] for k in sorted(expected)])
review_path=OUT/'blind_answer_review.csv'
editable=['reviewed_correct','reviewer_type','reviewer','review_note','needs_adjudication','human_correct','human_reviewer','human_note']
if review_path.exists():
    current=pd.read_csv(review_path,keep_default_na=False,dtype=str)
    assert current.review_id.is_unique and set(current.review_id)==set(expected_review.review_id)
    current=current.set_index('review_id').loc[expected_review.review_id].reset_index()
    pd.testing.assert_frame_equal(current[list(expected_review.columns)],expected_review,check_dtype=False)
else:
    current=expected_review.copy()
    for col in editable:current[col]=''
    current.to_csv(review_path,index=False)
immutable(mapping,OUT/'review_mapping_private.json')
validity_path=OUT/'development_question_validity.csv'
if not validity_path.exists():
    validity=pd.DataFrame(dev)[['question_id','question','reference_answer']].copy()
    for col in ['decision','reason','reviewer']:validity[col]=''
    validity.to_csv(validity_path,index=False)
# Validate question identities without requiring completed decisions yet.
v=pd.read_csv(validity_path,keep_default_na=False,dtype=str)
assert v.question_id.is_unique and set(v.question_id)=={x['question_id'] for x in dev}
v=v.set_index('question_id').loc[[x['question_id'] for x in dev]].reset_index()
pd.testing.assert_frame_equal(v[['question_id','question','reference_answer']],pd.DataFrame(dev)[['question_id','question','reference_answer']],check_dtype=False)
review_zip=OUT/'llama31_8b_low_noise_uq_review_bundle.zip'
with zipfile.ZipFile(review_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in [review_path,validity_path]:z.write(p,p.name)
    z.writestr('README.txt','Review only the greedy predictions. Accept unambiguous aliases and harmless spelling variations; reject contradictions, refusals, insufficient specificity, and repetitions. Judge truncation by visible content. Hold invalid questions consistently across all conditions. Identify AI-assisted labels; do not claim a human audit without one. Preserve all IDs and original text. Fill review_note with rationale. Identical answers across conditions share this review. Return the two CSVs unchanged in structure. No condition mapping or uncertainty scores belong in this reviewer bundle.')
print('Distinct review answers:',len(current))
print('Review bundle:',review_zip)


## 3 · Sample answers and score saved greedy tokens
Load one Llama adapter at a time. Per-question caches preserve all raw outputs. An empty sampled answer stops for investigation; it is not resampled until a desirable answer appears. Length-limit hits remain included. If interrupted, restart and run Sections 1–3; completed questions are reused. No training is performed.

In [ ]:
from google.colab import userdata
from getpass import getpass
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    GenerationConfig,
    StoppingCriteria,
    StoppingCriteriaList,
    set_seed,
)
from peft import PeftModel, get_peft_model_state_dict

assert (
    torch.cuda.is_available()
    and "L4" in torch.cuda.get_device_name(0)
    and torch.cuda.is_bf16_supported()
)

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass("Hugging Face token (hidden): ")
assert HF_TOKEN


def prompt_ids(question):
    text = "\n\n".join(
        "Question: " + r["question"] + "\nAnswer: " + r["answer"].strip()
        for r in shots
    )
    text += "\n\nQuestion: " + question + "\nAnswer:"
    return [tokenizer.bos_token_id] + tokenizer.encode(
        text, add_special_tokens=False
    )


class StopAtNewline(StoppingCriteria):
    def __init__(self, start):
        self.start = start

    def __call__(self, input_ids, scores, **kwargs):
        return torch.tensor(
            [
                "\n" in tokenizer.decode(
                    input_ids[0, self.start:],
                    skip_special_tokens=True,
                )
            ],
            device=input_ids.device,
        )


def answer_prefix(generated, eos_ids):
    kept = []
    mixed = False

    for t in generated:
        if t in eos_ids:
            break

        previous = tokenizer.decode(kept, skip_special_tokens=True)
        candidate = tokenizer.decode(kept + [t], skip_special_tokens=True)

        if "\n" in candidate:
            before = candidate.split("\n", 1)[0]
            contribution = (
                before[len(previous):]
                if before.startswith(previous)
                else before
            )
            if contribution.strip():
                kept.append(t)
                mixed = True
            break

        kept.append(t)

    while kept:
        full = tokenizer.decode(kept, skip_special_tokens=True)
        shorter = tokenizer.decode(kept[:-1], skip_special_tokens=True)

        if full.strip() == shorter.strip():
            kept.pop()
        else:
            break

    return kept, mixed


@torch.inference_mode()
def score_greedy(model, item, saved, eos_ids):
    ids, mixed = answer_prefix(saved["generated_token_ids"], eos_ids)

    # Greedy-answer scoring still requires a nonempty answer.
    assert ids and saved["prediction"].strip(), (
        "Empty greedy answer needs investigation."
    )
    assert (
        tokenizer.decode(
            saved["generated_token_ids"],
            skip_special_tokens=True,
        ).split("\n", 1)[0].strip()
        == saved["prediction"]
    )

    prefix = prompt_ids(item["question"])
    x = torch.tensor([prefix + ids], device="cuda")

    logits = model(
        input_ids=x,
        attention_mask=torch.ones_like(x),
        use_cache=False,
    ).logits[
        0, len(prefix) - 1:len(prefix) - 1 + len(ids)
    ].float()

    assert torch.isfinite(logits).all()

    lp = torch.log_softmax(logits, dim=-1)
    prob = lp.exp()
    target = torch.tensor(ids, device="cuda")

    entropy = -(prob * lp).sum(-1).mean().item()
    nll = -lp.gather(1, target[:, None]).mean().item()

    assert math.isfinite(entropy) and math.isfinite(nll)

    return {
        "mean_token_entropy": entropy,
        "mean_answer_nll": nll,
        "scored_token_ids": ids,
        "mixed_answer_newline_token": mixed,
    }


@torch.inference_mode()
def sample_answer(model, item, gen_config, eos_ids):
    ids = prompt_ids(item["question"])
    x = torch.tensor([ids], device="cuda")

    out = model.generate(
        input_ids=x,
        attention_mask=torch.ones_like(x),
        generation_config=gen_config,
        stopping_criteria=StoppingCriteriaList(
            [StopAtNewline(len(ids))]
        ),
    )[0, len(ids):].tolist()

    raw = tokenizer.decode(out, skip_special_tokens=True)
    answer = raw.split("\n", 1)[0].strip()
    ended = bool(out) and out[-1] in eos_ids

    return {
        "text": answer,
        "raw_continuation": raw,
        "generated_token_ids": out,
        "empty": not bool(answer),
        "hit_length_limit": (
            len(out) >= 32 and not ended and "\n" not in raw
        ),
    }


def validate_samples(samples):
    # Empty strings are valid observed outputs, not missing samples.
    assert isinstance(samples, list) and len(samples) == 10
    for sample in samples:
        assert isinstance(sample["text"], str)
        assert isinstance(sample["raw_continuation"], str)
        assert isinstance(sample["generated_token_ids"], list)
        assert sample["empty"] == (not bool(sample["text"].strip()))


def compute_condition(c):
    global tokenizer

    path = records_path(c)
    cache = (
        read(path)
        if path.exists()
        else {"binding": BINDING, "condition": c, "records": []}
    )

    assert cache["binding"] == BINDING and cache["condition"] == c

    by_id = {r["question_id"]: r for r in cache["records"]}
    assert len(by_id) == len(cache["records"])
    assert set(by_id) <= {r["question_id"] for r in dev}

    for r in cache["records"]:
        assert r["greedy_hash"] == fp(greedy[c][r["question_id"]])
        if "samples" in r:
            validate_samples(r["samples"])

    if len(by_id) == len(dev) and all(
        "mean_answer_nll" in r
        and "mean_token_entropy" in r
        and len(r.get("samples", [])) == 10
        for r in cache["records"]
    ):
        print(c, ": using completed cache.")
        return

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID, revision=REVISION, token=HF_TOKEN
    )
    tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=REVISION,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        device_map={"": "cuda:0"},
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    model = PeftModel.from_pretrained(
        base, adapters[c], is_trainable=False
    ).eval()
    model.config.use_cache = True

    eos_ids = model.generation_config.eos_token_id or tokenizer.eos_token_id
    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]

    assert eos_ids == read(PRE / "tokenizer_identity.json")["eos_ids"]

    gen_config = GenerationConfig(
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        top_k=0,
        num_beams=1,
        max_new_tokens=32,
        eos_token_id=eos_ids,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        repetition_penalty=1.0,
        use_cache=True,
    )

    for item in tqdm(dev, desc=c):
        q = item["question_id"]

        if q not in by_id:
            by_id[q] = {
                "question_id": q,
                "greedy_hash": fp(greedy[c][q]),
            }
            cache["records"].append(by_id[q])

        row = by_id[q]

        # Never regenerate an existing sample set.
        if "samples" not in row:
            seed = int(
                hashlib.sha256(f"123:{q}".encode()).hexdigest()[:8],
                16,
            )
            set_seed(seed)
            row["samples"] = [
                sample_answer(model, item, gen_config, eos_ids)
                for _ in range(10)
            ]
            row["sampling_seed"] = seed
            save(cache, path)

        validate_samples(row["samples"])

        if (
            "mean_answer_nll" not in row
            or "mean_token_entropy" not in row
        ):
            row.update(score_greedy(model, item, greedy[c][q], eos_ids))
            save(cache, path)

    empty_count = sum(
        not sample["text"].strip()
        for row in cache["records"]
        for sample in row["samples"]
    )
    print(f"{c}: preserved {empty_count} empty sampled outputs.")

    del model, base
    gc.collect()
    torch.cuda.empty_cache()


for c in CONDITIONS:
    compute_condition(c)

del HF_TOKEN
print("Sampling and token scoring complete.")

## 4 · Cluster sampled meanings and calculate semantic entropy
The Llama model is unloaded before the NLI model is loaded. Every member of a cluster must entail the new answer in both directions. Identical strings share a cluster. Ten-sample cluster frequencies determine entropy using natural logarithms. No manual cluster repairs enter the primary pilot metrics.

In [ ]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification
NLI_DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
nli_tokenizer=AutoTokenizer.from_pretrained(NLI_ID,revision=NLI_REV)
nli_model=AutoModelForSequenceClassification.from_pretrained(NLI_ID,revision=NLI_REV,
    torch_dtype=torch.float32).to(NLI_DEVICE).eval()
nli_labels={int(i):label.lower() for i,label in nli_model.config.id2label.items()}
assert set(nli_labels.values())=={'entailment','neutral','contradiction'}


In [ ]:
@torch.inference_mode()
def compare_meanings(question, answer_a, answer_b):
    text_a = f"Question: {question}\nAnswer: {answer_a}"
    text_b = f"Question: {question}\nAnswer: {answer_b}"

    inputs = nli_tokenizer(
        [text_a, text_b],
        [text_b, text_a],
        padding=True,
        truncation=False,
        return_tensors="pt",
    ).to(NLI_DEVICE)

    if inputs["input_ids"].shape[1] > 512:
        raise ValueError("NLI pair too long; inspect before truncating.")

    logits = nli_model(**inputs).logits.float()
    probabilities = logits.softmax(dim=-1)
    predicted_ids = probabilities.argmax(dim=-1).tolist()

    forward_label = nli_labels[predicted_ids[0]]
    backward_label = nli_labels[predicted_ids[1]]

    return {
        "a_to_b": forward_label,
        "b_to_a": backward_label,
        "equivalent": (
            forward_label == "entailment"
            and backward_label == "entailment"
        ),
    }

import math

import math

def calculate_semantic_entropy(question, answers):
    if not answers:
        raise ValueError("Missing sampled answers.")
    if any(not isinstance(answer, str) for answer in answers):
        raise TypeError("Every sampled answer must be a string.")

    clusters = []
    cluster_ids = []
    pair_cache = {}

    def equivalent(a, b):
        a, b = a.strip(), b.strip()

        # Empty outputs belong together, separate from nonempty answers.
        # Do not pass empty outputs to the NLI model.
        if not a or not b:
            return a == b

        if a == b:
            return True

        key = tuple(sorted((a, b)))
        if key not in pair_cache:
            pair_cache[key] = compare_meanings(
                question, a, b
            )["equivalent"]

        return pair_cache[key]

    for answer in answers:
        answer = answer.strip()
        assigned_cluster = None

        for cluster_id, cluster in enumerate(clusters):
            distinct_members = list(dict.fromkeys(cluster))

            if all(
                equivalent(answer, member)
                for member in distinct_members
            ):
                assigned_cluster = cluster_id
                break

        if assigned_cluster is None:
            assigned_cluster = len(clusters)
            clusters.append([])

        clusters[assigned_cluster].append(answer)
        cluster_ids.append(assigned_cluster)

    probabilities = [
        len(cluster) / len(answers)
        for cluster in clusters
    ]
    entropy = -sum(p * math.log(p) for p in probabilities)

    return {
        "semantic_cluster_ids": cluster_ids,
        "semantic_clusters": clusters,
        "num_semantic_clusters": len(clusters),
        "discrete_semantic_entropy": entropy,
        "empty_sample_count": sum(
            not answer.strip() for answer in answers
        ),
        "empty_sample_policy": "separate_no_answer_cluster_v1",
    }

In [ ]:
for c in CONDITIONS:
    source=read(records_path(c));assert source['binding']==BINDING and len(source['records'])==200
    binding={'evaluation':BINDING,'records_hash':fh(records_path(c))}
    path=semantic_path(c)
    cache=read(path) if path.exists() else {'binding':binding,'records':[]}
    assert cache['binding']==binding
    by_id={r['question_id']:r for r in cache['records']}
    assert len(by_id)==len(cache['records']) and set(by_id)<={r['question_id'] for r in dev}
    questions={r['question_id']:r['question'] for r in dev}
    for r in tqdm(source['records'],desc='NLI '+c):
        q=r['question_id']
        if q not in by_id:
            result=calculate_semantic_entropy(questions[q],[s['text'] for s in r['samples']])
            row={'question_id':q,**result};cache['records'].append(row);by_id[q]=row;save(cache,path)
    assert len(cache['records'])==200
    inspection=[]
    for row in cache['records']:
        for i,cluster in enumerate(row['semantic_clusters']):
            inspection.append({'question_id':row['question_id'],'question':questions[row['question_id']],
                'cluster_id':i,'samples':json.dumps(cluster,ensure_ascii=False)})
    pd.DataFrame(inspection).to_csv(OUT/(c+'_cluster_inspection.csv'),index=False)
del nli_model;gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
print('Semantic scores complete; cluster inspection files stay separate from reviewer bundle.')


## 5 · Download review bundle and stop
Send this ZIP for correctness/validity review. It contains neither mappings nor uncertainty scores. The bundle was created in Section 2; saved score files remain on Drive. Do not run Section 6 until reviewed CSVs have been restored to this notebook’s output folder.

In [ ]:
from google.colab import files
review_zip=OUT/'llama31_8b_low_noise_uq_review_bundle.zip'
assert review_zip.exists()
print('Send:',review_zip)
files.download(str(review_zip))

cluster_zip=OUT/f'llama_low_noise_clusters_seed{SEED}_{TRAIN_VERSION}.zip'
with zipfile.ZipFile(cluster_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for c in CONDITIONS:
        path=OUT/(c+'_cluster_inspection.csv')
        assert path.exists(), f'Missing clusters: {c}; finish Section 4.'
        z.write(path,path.name)
    z.writestr('provenance.json',json.dumps({'seed':SEED,'training_version':TRAIN_VERSION,
        'training_path':str(TRAIN),'evaluation_identity':BINDING,'conditions':CONDITIONS},indent=2))
print('Also send cluster inspection:',cluster_zip)
files.download(str(cluster_zip))


## 6 · After review: metrics and development thresholds
After restarting, run Section 1 and this section only. No GPU model loading is required. All review gates must pass; missing reviews are not replaced with automatic EM. AI-assisted results remain provisional. Identical question/answer pairs share one label.

Report accuracy, error-positive AUROC/AP, whole-tie right-step AURC, zero-semantic-entropy error rate, and largest attainable development coverage at observed error ≤10%. Thresholds are selected and assessed on this development cohort, so these coverage/risk values are **in-sample descriptive**, not held-out guarantees. No test evaluation is performed.

The direct changes in semantic-minus-token/NLL AUROC advantage are descriptive one-seed point estimates; no significance claims or training-run generalization follow. All three σ values are compared with clipping only. Results are versioned by hashes of reviews and score inputs. Reviewer mappings never enter the exported report.

In [ ]:
from sklearn.metrics import roc_auc_score,average_precision_score
import matplotlib.pyplot as plt
v=pd.read_csv(OUT/'development_question_validity.csv',keep_default_na=False,dtype=str)
assert v.question_id.is_unique and set(v.question_id)=={x['question_id'] for x in dev}
v=v.set_index('question_id').loc[[x['question_id'] for x in dev]].reset_index()
pd.testing.assert_frame_equal(v[['question_id','question','reference_answer']],pd.DataFrame(dev)[['question_id','question','reference_answer']],check_dtype=False)
assert v.decision.isin(['include','hold']).all(), 'Complete include/hold decisions for every question.'
assert v.reviewer.str.strip().ne('').all()
assert v.loc[v.decision=='hold','reason'].str.strip().ne('').all()
eligible=v.loc[v.decision=='include','question_id'].tolist();assert eligible
review=pd.read_csv(OUT/'blind_answer_review.csv',keep_default_na=False,dtype=str)
assert review.review_id.is_unique
salt=read(OUT/'review_salt_private.json')['salt']
expected={};mapping=[]
for c in CONDITIONS:
    for item in dev:
        pred=greedy[c][item['question_id']]['prediction'];aliases=list(dict.fromkeys([item['reference_answer'],*item['accepted_answers']]))
        rid=fp([salt,[item['question_id'],item['question'],pred,item['reference_answer'],aliases]])
        expected[rid]={'question_id':item['question_id'],'question':item['question'],'prediction':pred,
            'reference_answer':item['reference_answer'],'accepted_answers':json.dumps(aliases,ensure_ascii=False)}
        mapping.append({'condition':c,'question_id':item['question_id'],'review_id':rid})
assert mapping==read(OUT/'review_mapping_private.json')
assert set(review.review_id)==set(expected)
labels={};audit=[]
for r in review.to_dict('records'):
    assert all(r[k]==value for k,value in expected[r['review_id']].items()), 'Original review content changed.'
    if r['question_id'] not in eligible:continue
    assert r['reviewed_correct'] in ['0','1'] and r['reviewer_type'] in ['human','AI-assisted']
    assert r['reviewer'].strip() and r['review_note'].strip()
    assert r['needs_adjudication'] in ['0','1']
    human=r['human_correct'].strip()
    if human:
        assert human in ['0','1'] and r['human_reviewer'].strip() and r['human_note'].strip()
    assert r['needs_adjudication']=='0' or human, 'Resolve flagged answers before reporting.'
    label=int(human or r['reviewed_correct']);labels[r['review_id']]=label
    audit.append({'review_id':r['review_id'],'initial_label':int(r['reviewed_correct']),'final_label':label,
        'human_checked':bool(human) or r['reviewer_type']=='human','reviewer_type':r['reviewer_type']})
status='human-reviewed development labels' if all(r['human_checked'] for r in audit) else 'provisional AI-assisted development labels; human audit incomplete'
inputs={'evaluation':BINDING,'review':fh(OUT/'blind_answer_review.csv'),'validity':fh(OUT/'development_question_validity.csv'),
        'score_files':{c:[fh(records_path(c)),fh(semantic_path(c))] for c in CONDITIONS}}
REPORT=OUT/('reports_'+fp(inputs)[:12]);REPORT.mkdir(exist_ok=True)
immutable(inputs,REPORT/'analysis_inputs.json')

def whole_tie_curve(scores,errors):
    order=np.argsort(scores,kind='stable');s=np.asarray(scores)[order];e=np.asarray(errors)[order]
    end=np.r_[np.flatnonzero(s[:-1]!=s[1:]),len(s)-1]
    counts=end+1;coverage=counts/len(s);risk=np.cumsum(e)[end]/counts
    aurc=float(np.sum(np.diff(np.r_[0.,coverage])*risk))
    return s[end],coverage,risk,aurc

outcomes=[];curves=[];thresholds=[];zero=[]
for c in CONDITIONS:
    record_cache=read(records_path(c));assert record_cache['binding']==BINDING
    semantic_cache=read(semantic_path(c));assert semantic_cache['binding']=={'evaluation':BINDING,'records_hash':fh(records_path(c))}
    records={r['question_id']:r for r in record_cache['records']};sem={r['question_id']:r for r in semantic_cache['records']}
    assert len(records)==len(record_cache['records'])==len(sem)==len(semantic_cache['records'])==200
    link={r['question_id']:r['review_id'] for r in mapping if r['condition']==c}
    y=np.array([1-labels[link[q]] for q in eligible]);acc=float(1-y.mean())
    for score in SCORES:
        x=np.array([sem[q][score] if score=='discrete_semantic_entropy' else records[q][score] for q in eligible])
        assert np.isfinite(x).all()
        auc=float(roc_auc_score(y,x)) if len(set(y))==2 else None
        ap=float(average_precision_score(y,x)) if len(set(y))==2 else None
        ts,cov,risk,aurc=whole_tie_curve(x,y)
        good=np.flatnonzero(risk<=.10)
        best=int(good[-1]) if len(good) else None
        chosen=None if best is None else float(ts[best])
        outcomes.append({'condition':c,'score':score,'n':len(y),'accuracy':acc,'error_prevalence':float(y.mean()),
            'AUROC':auc,'AP':ap,'AURC_tie_step':aurc,
            'development_coverage_at_10pct_risk':0. if best is None else float(cov[best]),
            'development_risk_at_selected_threshold':None if best is None else float(risk[best]),
            'accounted_epsilon':summaries[c]['accounted_epsilon'],'review_status':status})
        thresholds.append({'condition':c,'score':score,'threshold':chosen,'abstain_all':best is None,
            'risk_target':.10,'selection_scope':'same development data; exploratory','eligible_ids':eligible})
        curves.extend({'condition':c,'score':score,'threshold':float(t),'coverage':float(a),'risk':float(b)} for t,a,b in zip(ts,cov,risk))
    mask=np.array([sem[q]['discrete_semantic_entropy']==0 for q in eligible])
    zero.append({'condition':c,'zero_entropy_n':int(mask.sum()),'coverage':float(mask.mean()),
                 'risk':float(y[mask].mean()) if mask.any() else None})
frame=pd.DataFrame(outcomes)
lookup={(r['condition'],r['score']):r['AUROC'] for r in outcomes}
contrasts=[]
for c in CONDITIONS:
    if not c.startswith('dp_'):continue
    for score in SCORES:
        a=lookup[c,score];b=lookup['clipping_poisson',score]
        contrasts.append({'comparison':c+' minus clipping_poisson','score':score,'delta_AUROC':None if a is None or b is None else a-b})
advantages=[]
for c in CONDITIONS:
    for comparator in SCORES[:2]:
        vals=[lookup[c,'discrete_semantic_entropy'],lookup[c,comparator],lookup['clipping_poisson','discrete_semantic_entropy'],lookup['clipping_poisson',comparator]]
        advantages.append({'condition':c,'comparator':comparator,
            'semantic_advantage':None if any(v is None for v in vals) else vals[0]-vals[1],
            'change_vs_clipping':None if any(v is None for v in vals) else (vals[0]-vals[1])-(vals[2]-vals[3])})
frame.to_csv(REPORT/'outcomes.csv',index=False)
pd.DataFrame(curves).to_csv(REPORT/'risk_coverage_curves.csv',index=False)
pd.DataFrame(contrasts).to_csv(REPORT/'noise_vs_clipping_AUROC.csv',index=False)
pd.DataFrame(advantages).to_csv(REPORT/'semantic_advantage.csv',index=False)
pd.DataFrame(zero).to_csv(REPORT/'zero_entropy_diagnostics.csv',index=False)
pd.DataFrame(audit).to_csv(REPORT/'review_provenance.csv',index=False)
v.to_csv(REPORT/'question_validity.csv',index=False)
save(thresholds,REPORT/'development_thresholds.json')
for score in SCORES:
    fig,ax=plt.subplots(figsize=(7,4))
    for c in CONDITIONS:
        a=[r for r in curves if r['condition']==c and r['score']==score]
        ax.plot([r['coverage'] for r in a],[r['risk'] for r in a],marker='.',markersize=2,label=c)
    ax.axhline(.10,color='gray',linestyle='--');ax.set(xlabel='Coverage',ylabel='Observed error',title=score+' (development)',ylim=(0,1),xlim=(0,1))
    ax.legend(fontsize=7);fig.tight_layout();fig.savefig(REPORT/(score+'_risk_coverage.png'),dpi=150);plt.close(fig)
(REPORT/'README.txt').write_text(status+'\nSingle-seed development results. Threshold coverage/risk is selected and measured on the same data; no held-out guarantee. No significance inference. AURC uses whole ties and right-endpoint step integration. Plots connect attainable points only for readability. Epsilon is standalone research mechanism accounting, not a joint private release. Human review changes are recorded through versioned input hashes.')
report_zip=OUT/('llama31_8b_low_noise_uq_reports_'+fp(inputs)[:12]+'.zip')
with zipfile.ZipFile(report_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in REPORT.iterdir():
        if p.is_file():z.write(p,p.name)
display(frame)
print('Send:',report_zip)
from google.colab import files
files.download(str(report_zip))


### Human audit evidence

See the [human audit notebook](DP_UQ_Llama31_07_Human_Audit.ipynb)
for completed review files, reproducible checks, and saved results.

- Correctness audit: 50/50 agreement with the original AI labels.
- Semantic audit: two incorrect merges and two incorrect splits
  against the selecting strata, plus inconsistencies across runs.
- Correctness labels and reported metrics were unchanged.
- Targeted clustering corrections remain pending.

These sample audits do not make the full dataset human-reviewed.